### COT Report data scraper

https://github.com/NDelventhal/cot_reports

In [1]:
import pandas as pd
import cot_reports as cot
from matplotlib import pyplot as plt
import numpy as np
import re
#%matplotlib widget
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML


### <u> 1. Instructions for using cot_reports </u>


From https://github.com/NDelventhal/cot_reports

#### Example: cot_hist()
df = cot.cot_hist(cot_report_type= 'traders_in_financial_futures_futopt')
#### cot_hist() downloads the historical bulk file for the specified report type, in this example the Traders in Financial Futures Futures-and-Options Combined report. Returns the data as dataframe.

#### Example: cot_year()
df = cot.cot_year(year = 2020, cot_report_type = 'traders_in_financial_futures_fut')
#### cot_year() downloads the single year file of the specified report type and year. Returns the data as dataframe.

#### Example for collecting data of a few years, here from 2017 to 2020, of a specified report:
df = pd.DataFrame()
begin_year = 2017
end_year = 2020
for i in range(begin_year, end_year + 1):
    single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_futopt')) 
    df = df.append(single_year, ignore_index=True)

#### Example: cot_all()
df = cot.cot_all(cot_report_type='legacy_fut')
#### cot_all() downloads the historical bulk file and all remaining single year files of the specified report type.  Returns the data as dataframe.

### 2. Download and Compile COT Data into a pandas dataframe

In [2]:
def cot_reader (start, end):
    df_list = []
    begin_year = start
    end_year = end
    for i in range(begin_year, end_year + 1):
        single_year = pd.DataFrame(cot.cot_year(i, cot_report_type='legacy_fut')) 
        df_list.append(single_year)
    
    df = pd.concat(df_list, ignore_index=True)
    
    df.rename(columns = {"Market and Exchange Names" : "Market", 
                     "As of Date in Form YYMMDD" : "Datetime", 
                     "Open Interest (All)" : "OI", 
                     "As of Date in Form YYYY-MM-DD" : "Date"}, inplace=True )
    
    df["Datetime"] = pd.to_datetime(df["Datetime"], format = '%y%m%d')
    
    df.sort_values("Datetime", ascending = False, inplace = True)
    
    return df

In [3]:
#try legacy_fut
#try all instead of year

In [4]:
df = cot_reader(2023, 2026)

Selected: legacy_fut
Downloaded single year data from: 2023
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2024
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2025
Stored the file annual.txt in the working directory.
Selected: legacy_fut
Downloaded single year data from: 2026
Stored the file annual.txt in the working directory.


### 3. Create a list of markets to trade 

##### This is based on personal preference, please ignore if you're looking at all markets or edit as you like

In [5]:
keywords = ["gold", "silver", "platinum", "palladium", "copper", "Lithium", "crude", "heating", "oil", "Nat Gas",
"rbob", "brent", "cocoa", "corn", "oat", "wheat", "soybean", "soy bean", "feed" , "Hogs", "live", "OJ", "coffee", "cotton", "sugar","E-Mini", "Micro","WTI-PHYSICAL",
    "Russell",
    "S&P 500",
    "NASDAQ",
    "Dow Jones",
    "Nikkei",
    "FTSE",
    "DAX",
    "CAC",
    "SMI",
    "Hang Seng",
    "Shanghai",
    "Treasury", "UST", "Bond", "EURO" , "Peso", "Brazilian", "Swiss", "Canadian", "British", "Japanese", "New Zealand", "Rand", 
    "bitcoin" , "ether","SOFR", "vix"
           ]

In [6]:
unique_markets = []

for keyword in keywords:
    filtered_df = df[df["Market"].str.contains(keyword, case=False)]
    
    unique_values = filtered_df["Market"].unique()
    
    unique_markets.extend(unique_values)
    

In [7]:
remove_items = [
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    'EURO SHORT TERM RATE - CHICAGO MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTY-PAID - COMMODITY EXCHANGE INC.',
    'EURODOLLARS-3M - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH EURODOLLARS - CHICAGO MERCANTILE EXCHANGE',
    'COFFEE CALENDAR SPREAD OPTIONS - ICE FUTURES U.S.',
    'WHEAT-HRW - CHICAGO BOARD OF TRADE',
    'WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE',
    'BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF TRADE',
    'CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE',
    'CORN CSO - CHICAGO BOARD OF TRADE',
    'MARINE .5% FOB USGC/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'WTI-BRENT SPREAD OPTION - NEW YORK MERCANTILE EXCHANGE',
    'WTI-BRENT CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'USGC HSFO-PLATTS/BRENT 1ST LN - ICE FUTURES ENERGY DIV',
    'TRANSCONTINENTAL GAS- STATION 85 (ZONE 4) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - VENTURA (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-TEXOK (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (BASIS) - ICE FUTURES ENERGY DIV',
    'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COLUMBIA GAS CO. - TCO POOL (APPALACHIA) (BASIS) - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PIPELINE-MID-CONTINENT POOL PIN (BASIS) - ICE FUTURES ENERGY DIV',
    'NORTHERN NATURAL GAS - DEMARCATION POOL (BASIS) - ICE FUTURES ENERGY DIV',
    'PANHANDLE EASTERN- POOL GAS (INDEX) - ICE FUTURES ENERGY DIV',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'ONEOK GAS TRANSPORTATION BASIS - ICE FUTURES ENERGY DIV',
    'NATURAL GAS INDEX: EP SAN JUAN - ICE FUTURES ENERGY DIV',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'NATURAL GAS HENRY LD1 FIXED - ICE FUTURES ENERGY DIV',
    'NATURAL GAS PENULTIMATE ICE - ICE FUTURES ENERGY DIV',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
    'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE DIFF-TMX WCS 1A INDEX - ICE FUTURES ENERGY DIV',
    'CRUDE DIFF-TMX SW 1A INDEX - ICE FUTURES ENERGY DIV',
    'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE OIL CAL SPREAD OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM - ICE FUTURES ENERGY DIV',
    'MT BELV NAT GASOLINE OPIS - NEW YORK MERCANTILE EXCHANGE',
     'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P FINANCIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P UTILITIES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P 400 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P ENERGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P CONSU STAPLES INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P TECHNOLOGY INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P HEALTH CARE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE', 
     'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 ANNUAL DIVIDEND INDEX - CHICAGO MERCANTILE EXCHANGE',
     'S&P 500 QUARTERLY DIVIDEND IND - CHICAGO MERCANTILE EXCHANGE', 
     'DOW JONES U.S. REAL ESTATE IDX - CHICAGO BOARD OF TRADE',
     'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',  
     'NIKKEI STOCK AVERAGE YEN DENOM - CHICAGO MERCANTILE EXCHANGE',
     'COLUMBIA GULF TRANSMISSION CO. -  MAINLINE POOL - ICE FUTURES ENERGY DIV',
     'PACIFIC GAS TRANSMISSION - MALIN (BASIS) - ICE FUTURES ENERGY DIV',
    'COPPER-GRADE #1 - COMMODITY EXCHANGE INC.',
    'LITHIUM HYDROXIDE - COMMODITY EXCHANGE INC.',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE EXCHANGE',
    'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
    'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MARINE FUEL OIL 0.5% FOB USGC - ICE FUTURES ENERGY DIV',
    'GULF JET NY HEAT OIL SPR - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV',
 'USD MALAYSIAN CRUDE PALM OIL - CHICAGO MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'FUEL OIL-3% USGC/3.5% FOB RDAM BARGES - ICE FUTURES ENERGY DIV',
    'EUR STYLE NATURAL GAS OPTIONS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GAS LD1 for GDD -TEXOK - ICE FUTURES ENERGY DIV',
 'NAT GAS ICE LD1 - ICE FUTURES ENERGY DIV',
 'NATURAL GAS CAL SPREAD OPT FIN - NEW YORK MERCANTILE EXCHANGE',
 'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
 'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
 'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
 'BRENT LAST DAY - NEW YORK MERCANTILE EXCHANGE',
 'BRENT CRUDE OIL LAST DAY - NEW YORK MERCANTILE EXCHANGE',
    'MICRO GOLD - COMMODITY EXCHANGE INC.',
    'WTI 1st Line-Brent 1st Line - ICE FUTURES ENERGY DIV',
    'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P MATERIALS INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 TOTAL RETURN INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE',
    'NFX CS5TC CAPESIZE 5T/C AVG - NASDAQ FUTURES',
    'NFX PM4TC PANAMAX 4T/C AVG - NASDAQ FUTURES',
    'NORTHWEST PIPELINE - CANADIAN BORDER (BASIS) - ICE FUTURES ENERGY DIV',
    'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
    'CRUDE DIFF-WCS CUSHING/WTI 1ST - ICE FUTURES ENERGY DIV',
    'DUTCH TTF NAT GAS CAL MONTH - NEW YORK MERCANTILE EXCHANGE',
    'TRANSCONTINENTAL GAS - ZONE 6 (NY) (BASIS) - ICE FUTURES ENERGY DIV',
    'GULF COAST UNL 87 GAS M2 PL RB - NEW YORK MERCANTILE EXCHANGE',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS INDEX: ALGONQUIN CITY GATES - ICE FUTURES ENERGY DIV',
    'HOT ROLLED COIL STEEL - NEW YORK MERCANTILE EXCHANGE',
    '3.5% FUEL OIL RDAM CRACK SPR - NEW YORK MERCANTILE EXCHANGE',
    'HENRY HUB PENULTIMATE NAT GAS - NEW YORK MERCANTILE EXCHANGE',
    'NAT GASLNE OPIS MT B NONTET FP - ICE FUTURES ENERGY DIV',
     'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'ULTRA U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'SOUTH AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE',
    'Nano Bitcoin - LMX LABS LLC',
    'NANO ETHER - LMX LABS LLC',
    '2 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    '#2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'WCS OIL NET ENERGY MONTHLY IND - NEW YORK MERCANTILE EXCHANGE',
    'NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
    'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'WTI  HOUSTON ARGUS/WTI TR MO - NEW YORK MERCANTILE EXCHANGE',
    'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
    'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NASDAQ-100 STOCK INDEX (MINI) - CHICAGO MERCANTILE EXCHANGE',
    'HOUSTON SHIP CHANNEL (INDEX) - ICE FUTURES ENERGY DIV',
    'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE',
     'HHUB NAT GAS PENULT FINL-10000 - NASDAQ FUTURES',
     'NAT GAS ICE PEN - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
     'E-MINI S&P INDUSTRIAL INDEX - CHICAGO MERCANTILE EXCHANGE',
     'ERCOT Houston 345KV Hub RT 7x8 - ICE FUTURES ENERGY DIV',
    'ERCOT Houston 345KV RT OFF FIX - ICE FUTURES ENERGY DIV',
    'ERCOT HOUSTON 345KV RT PK FIX - ICE FUTURES ENERGY DIV',
     'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
 'CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE',
 'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
  'GULF # 6 FUEL OIL CRACK - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) (BASIS) - ICE FUTURES ENERGY DIV',
 'TEXAS EASTERN- M3 ZONE (DELIVERED) - ICE FUTURES ENERGY DIV',
 "WAHA HUB - WEST TEXAS DELIVERED/BUYER'S INDEX - ICE FUTURES ENERGY DIV",
 '5 YEAR DELIVERABLE IR - CHICAGO BOARD OF TRADE',
 'MICRO 10 YEAR YIELD - CHICAGO BOARD OF TRADE', 
 'MICRO E-MINI DJIA (x$0.5) - CHICAGO BOARD OF TRADE',
 'MICRO SING FOB MARINE FUEL .5% - NEW YORK MERCANTILE EXCHANGE',
 '10 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 VALUE INDEX - CHICAGO MERCANTILE EXCHANGE',
 'E-MINI RUSSELL 2000 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'MICRO E-MINI S&P 500 INDEX - CHICAGO MERCANTILE EXCHANGE',
 'CRUDE DIFF-WCS HOUSTON/WTI 1ST - ICE FUTURES ENERGY DIV',
 '5 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
 'DOW JONES INDUSTRIAL AVG- x $5 - CHICAGO BOARD OF TRADE',
 'ARGUS WTI HOUSTON/WTI TRADE MO - ICE FUTURES ENERGY DIV',
 'U.S. TREASURY BONDS - CHICAGO BOARD OF TRADE',
 '10-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '5-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 '2-YEAR U.S. TREASURY NOTES - CHICAGO BOARD OF TRADE',
 'UST BOND - CHICAGO BOARD OF TRADE',
 'ULTRA UST 10Y - CHICAGO BOARD OF TRADE',
 'MICRO E-MINI RUSSELL 2000 INDX - CHICAGO MERCANTILE EXCHANGE',
 'NASDAQ MINI - CHICAGO MERCANTILE EXCHANGE',
 'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
 'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
     'ADJUSTED INT RATE S&P 500 TOTL - CHICAGO MERCANTILE EXCHANGE',
    'GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE',
    'GASOLINE CRK-RBOB/BRENT 1st - ICE FUTURES ENERGY DIV',
    'GULF COAST CBOB GAS A2 PL RBOB - NEW YORK MERCANTILE EXCHANGE',
    'RBOB CALENDAR - NEW YORK MERCANTILE EXCHANGE',
    'RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV',
    'FUEL OIL USGC HSFO PLATTS BALM - ICE FUTURES ENERGY DIV',
    'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE',
    'CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE',
    'E-MINI S&P REAL ESTATE INDEX - CHICAGO MERCANTILE EXCHANGE',
    'E-MINI S&P 500 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE',
    'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
    'EUROPEAN PROPANE CIF ARA - NEW YORK MERCANTILE EXCHANGE',
    'ALUMINIUM EURO PREM DUTYUNPAID - COMMODITY EXCHANGE INC.',
    'E-MINI S&P COMMUNICATION INDEX - CHICAGO MERCANTILE EXCHANGE',
    'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
    'RANDOM LENGTH LUMBER - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-3M - CHICAGO MERCANTILE EXCHANGE',
    'SOFR-1M - CHICAGO MERCANTILE EXCHANGE',
    '1-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    '3-MONTH SOFR - CHICAGO MERCANTILE EXCHANGE',
    'NANO BITCOIN PERP STYLE - COINBASE DERIVATIVES, LLC',
    'NANO ETHER - COINBASE DERIVATIVES, LLC',
    'NANO ETHER PERP STYLE - COINBASE DERIVATIVES, LLC',
'NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.',
'Nano Bitcoin - COINBASE DERIVATIVES, LLC',
'RUSSELL 2000 ANNUAL DIVIDEND - CHICAGO MERCANTILE EXCHANGE',
'ULTRA US T BOND - CHICAGO BOARD OF TRADE',
'ULTRA UST BOND - CHICAGO BOARD OF TRADE',
'WHEAT-HRSpring - MIAX FUTURES EXCHANGE',
'WTI  HOUSTON ARGUS/WTI BALMO - NEW YORK MERCANTILE EXCHANGE',
'3 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE',
'EURO FX - CHICAGO MERCANTILE EXCHANGE',
'EURO FX/JAPANESE YEN XRATE - CHICAGO MERCANTILE EXCHANGE',
'MICRO SOL - CHICAGO MERCANTILE EXCHANGE',
'MICRO XRP - CHICAGO MERCANTILE EXCHANGE'





    
    
]


In [8]:
for item in remove_items:
    if item in unique_markets:
        unique_markets.remove(item)

In [9]:
sorted(list(dict.fromkeys(unique_markets))) #this is the market list for making graphs.

['AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE',
 'BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE',
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE',
 'COCOA - ICE FUTURES U.S.',
 'COFFEE C - ICE FUTURES U.S.',
 'COPPER- #1 - COMMODITY EXCHANGE INC.',
 'CORN - CHICAGO BOARD OF TRADE',
 'COTTON NO. 2 - ICE FUTURES U.S.',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE',
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE',
 'GOLD - COMMODITY EXCHANGE INC.',
 'GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC',
 'GRP 3 SOC GAS VS RBOB SPR - NEW YORK MERCANTILE EXCHANGE',
 'JAPANES

In [10]:
#remove duplicates
unique_markets = list(dict.fromkeys(unique_markets))

### 4. Create Open Interest Index Value for a Commodity 

In [11]:
#create new DF for this part of the analysis
df2 = df.sort_values(['Market', 'Datetime'], ascending = [True, True])

In [12]:
#Group markerss and add Open Interest Index Column

group = df2.groupby("Market")["OI"]
df2["OI_Index"] = group.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

## Forumla for indexing:
#zi = (xi – min(x)) / (max(x) – min(x)) * 100 = (12 – 12) / (68 – 12) * 100 = 0


### 5. See if a market has a unique value based on your keyword or search term


In [13]:
def number_of_markets (keyword_or_phrase):
    
    filtered = df[df["Market"].str.contains(keyword_or_phrase, case=False)]
    filtered = filtered["Market"].unique().tolist()
    
    return filtered

In [14]:
number_of_markets("cocoa")

['COCOA - ICE FUTURES U.S.']

In [15]:
df2.iloc[-1]

Market                                   XRP - CHICAGO MERCANTILE EXCHANGE
Datetime                                               2026-03-10 00:00:00
Date                                                            2026-03-10
CFTC Contract Market Code                                           176740
CFTC Market Code in Initials                                          CME 
                                                       ...                
Contract Units                                   (Contracts of 50,000 XRP)
CFTC Contract Market Code (Quotes)                                  176740
CFTC Market Code in Initials (Quotes)                                 CME 
CFTC Commodity Code (Quotes)                                           176
OI_Index                                                         56.958539
Name: 52554, Length: 130, dtype: object

### Retail OI indexing

In [16]:
df2.columns.to_list()

['Market',
 'Datetime',
 'Date',
 'CFTC Contract Market Code',
 'CFTC Market Code in Initials',
 'CFTC Region Code',
 'CFTC Commodity Code',
 'OI',
 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
 'Open Interest (Old)',
 'Noncommercial Positions-Long (Old)',
 'Noncommercial Positions-Short (Old)',
 'Noncommercial Positions-Spreading (Old)',
 'Commercial Positions-Long (Old)',
 'Commercial Positions-Short (Old)',
 'Total Reportable Positions-Long (Old)',
 'Total Reportable Positions-Short (Old)',
 'Nonreportable Positions-Long (Old)',
 'Nonreportable Positions-Short (Old)',
 'Open Interest (Other)',
 'Noncommercial Positions-Long (Other)',
 'Noncommercial Positions-Short (Other)'

In [17]:
cols = ['Market', 'Datetime' , 'Noncommercial Positions-Long (All)',
 'Noncommercial Positions-Short (All)',
 'Noncommercial Positions-Spreading (All)',
 'Commercial Positions-Long (All)',
 'Commercial Positions-Short (All)',
 ' Total Reportable Positions-Long (All)',
 'Total Reportable Positions-Short (All)',
 'Nonreportable Positions-Long (All)',
 'Nonreportable Positions-Short (All)',
        'OI_Index', "OI"]

In [18]:
df3 = df2[cols].copy()

In [19]:
df3.iloc[0, 1:13]

Datetime                                   2023-08-15 00:00:00
Noncommercial Positions-Long (All)                       22834
Noncommercial Positions-Short (All)                      20587
Noncommercial Positions-Spreading (All)                  52779
Commercial Positions-Long (All)                              0
Commercial Positions-Short (All)                          2226
 Total Reportable Positions-Long (All)                   75613
Total Reportable Positions-Short (All)                   75592
Nonreportable Positions-Long (All)                           3
Nonreportable Positions-Short (All)                         24
OI_Index                                              4.363201
OI                                                       75616
Name: 15062, dtype: object

In [20]:
row_test = df3.iloc[0, 1:10] #list of column headers and 1st row of data

In [21]:
df3["Net Retail Position"] = df3["Nonreportable Positions-Long (All)"] - df3["Nonreportable Positions-Short (All)"]

In [22]:
group2 = df3.groupby("Market")["Net Retail Position"]
df3["Retail_Index"] = group2.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [23]:
df3["Net Commercial Position"] = df3["Commercial Positions-Long (All)"] - df3["Commercial Positions-Short (All)"]


In [24]:
group3 = df3.groupby("Market")["Net Commercial Position"]
df3["Commercial_Index"] = group3.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [25]:
df3["Net Traders Position"] = df3["Noncommercial Positions-Long (All)"] - df3["Noncommercial Positions-Short (All)"]


In [26]:
group4 = df3.groupby("Market")["Net Traders Position"]
df3["Traders_Index"] = group4.transform(lambda x: (((x - min(x)) / (max(x) - min(x))) * 100))

In [27]:
df3.tail()

,Market,Datetime,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Noncommercial Positions-Spreading (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Total Reportable Positions-Long (All),Total Reportable Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All),OI_Index,OI,Net Retail Position,Retail_Index,Net Commercial Position,Commercial_Index,Net Traders Position,Traders_Index
52558,XRP - CHICAGO MERCANTILE EXCHANGE,2026-02-10,6859,6152,9,70,646,6938,6807,111,242,43.418382,7049,-131,27.272727,-576,19.888734,707,82.234637
52557,XRP - CHICAGO MERCANTILE EXCHANGE,2026-02-17,6770,6031,73,37,646,6880,6750,106,236,42.505074,6986,-130,27.727273,-609,15.299026,739,85.810056
52556,XRP - CHICAGO MERCANTILE EXCHANGE,2026-02-24,6317,5613,904,43,646,7264,7163,165,266,48.927225,7429,-101,40.909091,-603,16.133519,704,81.899441
52555,XRP - CHICAGO MERCANTILE EXCHANGE,2026-03-03,5694,4860,478,74,717,6246,6055,50,241,32.502175,6296,-191,0.000000,-643,10.570236,834,96.424581
52554,XRP - CHICAGO MERCANTILE EXCHANGE,2026-03-10,6182,5316,1688,38,757,7908,7761,75,222,56.958539,7983,-147,20.000000,-719,0.000000,866,100.000000


### Summarise Latest Week's Data in a Table

In [28]:
OI_condition = df3[(df3['OI_Index'] >= 80) | (df3['OI_Index']<=20)]
Retail_condition = df3[(df3['Retail_Index'] >= 80) | (df3['Retail_Index']<=20)]
Commercial_condition = df3[(df3['Commercial_Index'] >= 80) | (df3['Commercial_Index']<=20)]
Date_coundition =df3["Datetime"].max() 

In [29]:
# Get the most recent date
most_recent_date = df3['Datetime'].max()

# Filter the DataFrame for the most recent date and conditions

summary_table = df3.loc[(df3['Datetime'] == most_recent_date) & 
                        (((df3['OI_Index'] >= 80) | (df3['OI_Index'] <= 20)) |
                         ((df3['Retail_Index'] >= 80) | (df3['Retail_Index'] <= 20)) |
                         ((df3['Commercial_Index'] >= 80) | (df3['Commercial_Index'] <= 20))),
                        ['Market', 'OI_Index', 'Retail_Index', 'Commercial_Index']]


In [30]:
summary_table_filtered = summary_table[summary_table['Market'].isin(unique_markets)]
summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_77532/3186465317.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table_filtered['Market'] = summary_table_filtered['Market'].str.split(' -').str[0].str.strip()


### Mapping symbols from yf to COT CFTC data

In [31]:
mapping = {'GOLD - COMMODITY EXCHANGE INC.' : 'GC=F',
 'SILVER - COMMODITY EXCHANGE INC.': 'SI=F',
 'PLATINUM - NEW YORK MERCANTILE EXCHANGE' : 'PL=F',
 'PALLADIUM - NEW YORK MERCANTILE EXCHANGE' : 'PA=F',
 'COPPER- #1 - COMMODITY EXCHANGE INC.' : "HG=F",
 'SOYBEAN OIL - CHICAGO BOARD OF TRADE' : "ZL=F",
 'SOYBEANS - CHICAGO BOARD OF TRADE': "ZS=F",
 'SOYBEAN MEAL - CHICAGO BOARD OF TRADE' : "ZM=F",
 'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE':'CL=F',
 'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE' : 'RB=F',
 'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE' : 'NG=F',
 'CORN - CHICAGO BOARD OF TRADE' : "ZC=F",
 'OATS - CHICAGO BOARD OF TRADE' : "ZO=F",
 'WHEAT-SRW - CHICAGO BOARD OF TRADE' : 'ZW=F',
 'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE' : "GF=F",
 'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE' : "HE=F",
 'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE' : "LE=F",
 'COCOA - ICE FUTURES U.S.' : "CC=F",
 'COFFEE C - ICE FUTURES U.S.' : "KC=F",
 'COTTON NO. 2 - ICE FUTURES U.S.' : "CT=F",
 'SUGAR NO. 11 - ICE FUTURES U.S.' : "SB=F",
 'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE' : 'RTY=F',
 'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE' : 'ES=F',
 'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'ETH-USD',
 'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE' : 'BTC-USD',
 'MICRO GOLD - COMMODITY EXCHANGE INC.': 'MGC=F',
 'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE' : 'RTY=F',
 'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE' : 'MNQ=F',
 'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE' : 'NKD=F',
 'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE' : '6A=F',
 'UST 5Y NOTE - CHICAGO BOARD OF TRADE' : 'ZF=F',
 'UST 2Y NOTE - CHICAGO BOARD OF TRADE' : 'ZT=F',
 'UST 10Y NOTE - CHICAGO BOARD OF TRADE' : 'ZN=F',
 'UST BOND - CHICAGO BOARD OF TRADE' : 'ZB=F',
 'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': '6M=F',
 'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE' : '6L=F',
 'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE' : '6S=F', 
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE' : '6C=F',
 'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE' : '6E=F',
 'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE' : '6B=F',
 'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE' : "6J=F",
 'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': '6N=F',
 'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE' : '6Z=F',
 'BITCOIN - CHICAGO MERCANTILE EXCHANGE' : "BTC=F",
 'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE' : 'ETH=F',
 'VIX FUTURES - CBOE FUTURES EXCHANGE' : '^VIX',
 'MINI SOYBEANS - CHICAGO BOARD OF TRADE': "ZS=F",
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': 'RTY=F',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': 'NG=F'
 }

In [32]:
#mapping.items maps keys and values in tuple pairs
#list()puts the tuples in a list
#pdDataframe puts that in a df, with said column names
mapping_df = pd.DataFrame(list(mapping.items()), columns=['Market', 'YF_Symbol'])

In [33]:
market_categories = {
    'GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'SILVER - COMMODITY EXCHANGE INC.': 'Metals',
    'PLATINUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'PALLADIUM - NEW YORK MERCANTILE EXCHANGE': 'Metals',
    'COPPER- #1 - COMMODITY EXCHANGE INC.': 'Metals',
    'SOYBEAN OIL - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
    'SOYBEAN MEAL - CHICAGO BOARD OF TRADE': 'Softs',
    'WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE': 'Energies',
    'CORN - CHICAGO BOARD OF TRADE': 'Grains',
    'OATS - CHICAGO BOARD OF TRADE': 'Grains',
    'WHEAT-SRW - CHICAGO BOARD OF TRADE': 'Grains',
    'FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LEAN HOGS - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'LIVE CATTLE - CHICAGO MERCANTILE EXCHANGE': 'Livestock',
    'COCOA - ICE FUTURES U.S.': 'Softs',
    'COFFEE C - ICE FUTURES U.S.': 'Softs',
    'COTTON NO. 2 - ICE FUTURES U.S.': 'Softs',
    'SUGAR NO. 11 - ICE FUTURES U.S.': 'Softs',
    'RUSSELL E-MINI - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'MICRO ETHER - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'MICRO GOLD - COMMODITY EXCHANGE INC.': 'Metals',
    'MICRO E-MINI NASDAQ-100 INDEX - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE': 'Indices',
    'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'UST 5Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 2Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST 10Y NOTE - CHICAGO BOARD OF TRADE': 'Financials',
    'UST BOND - CHICAGO BOARD OF TRADE': 'Financials',
    'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SWISS FRANC - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BRITISH POUND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'SO AFRICAN RAND - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
    'BITCOIN - CHICAGO MERCANTILE EXCHANGE': 'Currencies',
     'ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE' : 'Currencies',
    'VIX FUTURES - CBOE FUTURES EXCHANGE': 'Indices',
     'MINI SOYBEANS - CHICAGO BOARD OF TRADE': 'Softs',
 'EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE': 'Indices',
 'E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE': 'Energies'
}


In [34]:
#map the categories to the market in the mapping df
mapping_df["group"] = mapping_df["Market"].map(market_categories)

In [35]:
df4 = df3.copy()

In [36]:
df4 = pd.merge(df4, mapping_df, on="Market", how='left')

In [37]:
df4.rename(columns = {"Datetime" : "Date"}, inplace = True)

In [38]:
df4['group'].fillna('Financials', inplace=True)

/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_77532/638600763.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df4['group'].fillna('Financials', inplace=True)


## Create RSI Function


In [39]:
def rsi(data, periods=10):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

## Download Price Data

In [40]:
import yfinance as yf
from datetime import date, timedelta
start_date="2022-01-01"
end_date = date.today().strftime("%Y-%m-%d")
tickers = mapping_df['YF_Symbol'].to_list()
commods = yf.download(tickers, start=start_date, end=end_date)




[*********************100%***********************]  45 of 45 completed


In [41]:
# Create daily price dataset for all markets - INCLUDING OHLC for ATR calculation
daily_prices = []
for market_name in unique_markets:
    # Get the YF_Symbol for this market
    market_symbol = mapping_df[mapping_df['Market'] == market_name]['YF_Symbol'].iloc[0] if len(mapping_df[mapping_df['Market'] == market_name]) > 0 else None
    
    if market_symbol and market_symbol in commods['Close'].columns:
        try:
            # Start with Close price
            market_daily = commods['Close'][market_symbol].reset_index()
            market_daily.columns = ['Date', 'Close']
            
            # Add Open, High, Low for ATR calculation
            for price_col in ['Open', 'High', 'Low', 'Volume']:
                if price_col in commods.columns and market_symbol in commods[price_col].columns:
                    market_daily[price_col] = commods[price_col][market_symbol].values
            
            market_daily['Market'] = market_name
            market_daily['YF_Symbol'] = market_symbol
            daily_prices.append(market_daily)
            print(f"✓ {market_name}: {list(market_daily.columns)}")
        except Exception as e:
            print(f"✗ Error processing {market_name}: {e}")

# Combine all daily prices
daily_price_df = pd.concat(daily_prices, ignore_index=True)
# NOTE: dropna moved to after diagnostic cell


print(f"\n✓ Daily price data columns: {list(daily_price_df.columns)}")
print(f"✓ Total records (before cleaning): {len(daily_price_df)}")

✓ GOLD - COMMODITY EXCHANGE INC.: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ SILVER - COMMODITY EXCHANGE INC.: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ PLATINUM - NEW YORK MERCANTILE EXCHANGE: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ PALLADIUM - NEW YORK MERCANTILE EXCHANGE: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ COPPER- #1 - COMMODITY EXCHANGE INC.: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ SOYBEAN OIL - CHICAGO BOARD OF TRADE: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ NAT GAS NYME - NEW YORK MERCANTILE EXCHANGE: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Market', 'YF_Symbol']
✓ COCOA - ICE FUTURES U.S.: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume

In [42]:
# DIAGNOSTIC: Identify markets with missing price data (only unique_markets)
print("=" * 80)
print("PRICE DATA DIAGNOSTIC - Missing Data by Market")
print("=" * 80)

# Check which markets in unique_markets have no price data at all
target_markets = set(unique_markets)
price_markets = set(daily_price_df['Market'].unique())
missing_entirely = target_markets - price_markets

print(f"\n🔴 Markets in unique_markets but NO price data ({len(missing_entirely)}):")
for m in sorted(missing_entirely):
    print(f"   - {m}")

# Check for markets with partial/null price data (only unique_markets)
print(f"\n🟡 Markets with incomplete OHLC data:")
# Filter to only unique_markets
df_filtered = daily_price_df[daily_price_df['Market'].isin(unique_markets)]
null_summary = df_filtered.groupby('Market').agg({
    'Close': lambda x: x.isnull().sum(),
    'Open': lambda x: x.isnull().sum(),
    'High': lambda x: x.isnull().sum(),
    'Low': lambda x: x.isnull().sum(),
    'Volume': lambda x: x.isnull().sum()
}).rename(columns={'Close': 'Close_nulls', 'Open': 'Open_nulls', 
                   'High': 'High_nulls', 'Low': 'Low_nulls', 'Volume': 'Volume_nulls'})

# Also get total rows per market
null_summary['Total_Rows'] = df_filtered.groupby('Market').size()

# Show markets with any null values
markets_with_nulls = null_summary[
    (null_summary['Close_nulls'] > 0) | 
    (null_summary['Open_nulls'] > 0) | 
    (null_summary['High_nulls'] > 0) | 
    (null_summary['Low_nulls'] > 0)
]

if len(markets_with_nulls) > 0:
    print(markets_with_nulls.to_string())
else:
    print("   ✓ All markets have complete OHLC data!")

# Summary stats
print(f"\n📊 Summary:")
print(f"   Total unique_markets: {len(target_markets)}")
print(f"   Markets with price data: {len(price_markets)}")
print(f"   Markets missing entirely: {len(missing_entirely)}")
print("=" * 80)


PRICE DATA DIAGNOSTIC - Missing Data by Market

🔴 Markets in unique_markets but NO price data (6):
   - BITCOIN CASH PERP STYLE - COINBASE DERIVATIVES, LLC
   - GOLD -1 TROY OUNCE - COINBASE DERIVATIVES, LLC
   - GRP 3 SOC GAS VS RBOB SPR - NEW YORK MERCANTILE EXCHANGE
   - MICRO COPPER - COMMODITY EXCHANGE INC.
   - MICRO SILVER - COMMODITY EXCHANGE INC.
   - NAT GAS TETCO-WLA INDEX - ICE FUTURES ENERGY DIV

🟡 Markets with incomplete OHLC data:
                                                             Close_nulls  Open_nulls  High_nulls  Low_nulls  Volume_nulls  Total_Rows
Market                                                                                                                               
AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE                      481         481         481        481           480        1540
BITCOIN - CHICAGO MERCANTILE EXCHANGE                                481         481         481        481           480        1540
BRAZILIAN REAL

#### Add Price data to COT dataframe

In [43]:
df5 = pd.merge(df4, daily_price_df[['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume']], on=['Market', 'Date'], how='left')

##### df 5 now contains price and open interest by week but for all markets, not just the unique_markets

In [44]:
df5 = df5.sort_values(['Market', 'Date'], ascending = [True, True])

##### df6 has weekly COT and price data along with RSI but does not have daily prices

In [45]:
#Put the data above into a dataframe without the COT data for markets we are not trading (i.e removing everything not in unique markets)

df6 = []  # Initialize an empty list to store filtered dataframes

for market_name in unique_markets:
    df6.append(df5[df5["Market"] == market_name])

# Concatenate the filtered dataframes into a single DataFrame
df6 = pd.concat(df6, ignore_index=True)


        

In [46]:
# Calculate RSI for daily price data
daily_price_df['RSI'] = daily_price_df.groupby('Market')['Close'].transform(lambda x: rsi(x))
# Create a combined dataset that includes both weekly COT data and daily price data
# We'll add a 'data_type' column to distinguish between weekly COT data and daily price data

# Add data_type to existing df6 (weekly COT data)
df6_weekly = df6.copy()
df6_weekly['data_type'] = 'weekly_cot'

# Prepare daily price data to match df6 structure
daily_price_expanded = daily_price_df.copy()
daily_price_expanded['data_type'] = 'daily_price'

# Add missing columns from df6 to daily_price_expanded (fill with NaN for COT-specific data)

cot_columns = ['OI', 'OI_Index', 'Retail_Index', 'Commercial_Index', 'Traders_Index', 
               'Net Retail Position', 'Net Commercial Position', 'Net Traders Position',
               'Noncommercial Positions-Long (All)', 'Noncommercial Positions-Short (All)',
               'Commercial Positions-Long (All)', 'Commercial Positions-Short (All)',
               'Nonreportable Positions-Long (All)', 'Nonreportable Positions-Short (All)']
               

for col in cot_columns:
    if col not in daily_price_expanded.columns:
        daily_price_expanded[col] = None

# Add group column to daily price data by merging with mapping_df
if 'group' not in daily_price_expanded.columns:
    daily_price_expanded = pd.merge(daily_price_expanded, mapping_df[['Market', 'group']], on='Market', how='left')

# Add missing columns from daily data to df6_weekly if needed
if 'YF_Symbol' not in df6_weekly.columns:
    # Merge YF_Symbol from mapping_df
    df6_weekly = pd.merge(df6_weekly, mapping_df[['Market', 'YF_Symbol']], on='Market', how='left')

# Ensure both dataframes have the same column structure - NOW INCLUDES OHLC
common_columns = ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'YF_Symbol', 'data_type', 'group'] + cot_columns

# Reorder columns for consistency
df6_weekly = df6_weekly.reindex(columns=common_columns, fill_value=None)
daily_price_expanded = daily_price_expanded.reindex(columns=common_columns, fill_value=None)

print(f"✓ Common columns now include OHLC: {common_columns[:12]}...")

# Combine the datasets
df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)
df6_combined = df6_combined.sort_values(['Market', 'Date']).reset_index(drop=True)

print(f"Combined dataset created with {len(df6_combined)} total records")
print(f"Weekly COT data: {len(df6_weekly)} records")
print(f"Daily price data: {len(daily_price_expanded)} records")


✓ Common columns now include OHLC: ['Market', 'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'RSI', 'YF_Symbol', 'data_type', 'group', 'OI']...
Combined dataset created with 79580 total records
Weekly COT data: 7200 records
Daily price data: 72380 records


/var/folders/km/x_cb0wn5249ddhfzzt066kg40000gn/T/ipykernel_77532/2032221804.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df6_combined = pd.concat([df6_weekly, daily_price_expanded], ignore_index=True)


In [47]:
df6_combined = df6_combined.dropna(subset=['Close'])

In [48]:
# Filter for March 9th, 2026 only
df_march_9 = df6_combined[df6_combined['Date'] == '2026-03-10']

markets_with_close = df_march_9[df_march_9['Close'].notna()]['Market'].unique()

print(f"Total markets with close prices on March 10th: {len(markets_with_close)}")
print("\nMarkets with close prices on March 10th:")
for market in sorted(markets_with_close):
    print(f"  - {market}")

Total markets with close prices on March 10th: 47

Markets with close prices on March 10th:
  - AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - BITCOIN - CHICAGO MERCANTILE EXCHANGE
  - BRAZILIAN REAL - CHICAGO MERCANTILE EXCHANGE
  - BRITISH POUND - CHICAGO MERCANTILE EXCHANGE
  - CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE
  - COCOA - ICE FUTURES U.S.
  - COFFEE C - ICE FUTURES U.S.
  - COPPER- #1 - COMMODITY EXCHANGE INC.
  - CORN - CHICAGO BOARD OF TRADE
  - COTTON NO. 2 - ICE FUTURES U.S.
  - E-MINI NATURAL GAS - NEW YORK MERCANTILE EXCHANGE
  - E-MINI S&P 500 - CHICAGO MERCANTILE EXCHANGE
  - EMINI RUSSELL 1000 GROWTH - CHICAGO MERCANTILE EXCHANGE
  - ETHER CASH SETTLED - CHICAGO MERCANTILE EXCHANGE
  - EURO FX/BRITISH POUND XRATE - CHICAGO MERCANTILE EXCHANGE
  - FEEDER CATTLE - CHICAGO MERCANTILE EXCHANGE
  - GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE
  - GOLD - COMMODITY EXCHANGE INC.
  - JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE
  - LEAN HOGS - CHICAGO MERCANTILE EXC

In [49]:
platinum = df6_combined[df6_combined["Market"] == "SUGAR NO. 11 - ICE FUTURES U.S."]
platinum.sort_values("Date", ascending = False)

,Market,Date,Close,Open,High,Low,Volume,RSI,YF_Symbol,data_type,...,Traders_Index,Net Retail Position,Net Commercial Position,Net Traders Position,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All)
65922,SUGAR NO. 11 - ICE FUTURES U.S.,2026-03-19,15.370000,14.800000,15.490000,14.760000,115340.0,78.915650,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
65921,SUGAR NO. 11 - ICE FUTURES U.S.,2026-03-18,14.800000,14.390000,14.840000,14.350000,115340.0,67.889906,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
65920,SUGAR NO. 11 - ICE FUTURES U.S.,2026-03-17,14.450000,14.210000,14.510000,14.190000,74666.0,52.702697,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
65919,SUGAR NO. 11 - ICE FUTURES U.S.,2026-03-16,14.190000,14.410000,14.440000,14.150000,58213.0,27.083329,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
65916,SUGAR NO. 11 - ICE FUTURES U.S.,2026-03-13,14.370000,14.400000,14.530000,14.300000,59593.0,47.663544,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64223,SUGAR NO. 11 - ICE FUTURES U.S.,2022-01-07,18.049999,18.260000,18.440001,17.990000,46856.0,NaN,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
64222,SUGAR NO. 11 - ICE FUTURES U.S.,2022-01-06,18.190001,18.309999,18.389999,18.150000,46247.0,NaN,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
64221,SUGAR NO. 11 - ICE FUTURES U.S.,2022-01-05,18.340000,18.799999,18.799999,18.320000,59414.0,NaN,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None
64220,SUGAR NO. 11 - ICE FUTURES U.S.,2022-01-04,18.750000,18.750000,18.850000,18.620001,40168.0,NaN,SB=F,daily_price,...,NaN,None,None,None,None,None,None,None,None,None


In [50]:
# Export the combined dataset (both weekly COT data and daily price data)
df6_combined.to_json('cot_data.json', orient='records')
print("Combined data (weekly COT + daily prices) exported to cot_data.json")
print(f"Total records exported: {len(df6_combined)}")

Combined data (weekly COT + daily prices) exported to cot_data.json
Total records exported: 57882


In [51]:
# =============================================================================
# 🔔 CHECK FOR RECENT TRADING SIGNALS (Last 7 Days)
# =============================================================================
from datetime import datetime, timedelta

def check_recent_signals(df, days=7, commercial_long=80, commercial_short=20, 
                         rsi_oversold=30, rsi_overbought=70):
    """Check for trading signals in the last N days."""
    cutoff_date = datetime.now() - timedelta(days=days)
    recent_signals = []
    
    markets = df['Market'].unique()
    
    for market in markets:
        market_data = df[df['Market'] == market].copy()
        price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()
        cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
        
        if price_daily.empty or cot_weekly.empty:
            continue
        
        # Check if Commercial_Index exists in COT data
        if 'Commercial_Index' not in cot_weekly.columns:
            continue
            
        # Sort both by Date for merge_asof
        price_daily = price_daily[['Date', 'Close', 'RSI']].copy().sort_values('Date').reset_index(drop=True)
        cot_for_merge = cot_weekly[['Date', 'Commercial_Index']].dropna(subset=['Commercial_Index']).copy()
        cot_for_merge = cot_for_merge.sort_values('Date').reset_index(drop=True)
        
        if cot_for_merge.empty:
            continue
        
        # Use merge_asof to carry forward most recent COT data to each daily price row
        # direction='backward' means: find most recent COT date <= each price date
        merged = pd.merge_asof(price_daily, cot_for_merge, on='Date', direction='backward')
        
        # Drop rows with missing required data
        merged = merged.dropna(subset=['Close', 'RSI', 'Commercial_Index'])
        
        # Filter to recent days
        merged = merged[merged['Date'] >= cutoff_date]
        
        for _, row in merged.iterrows():
            signal_type = None
            # Long: Commercial >= 80 AND RSI < 30
            if row['Commercial_Index'] >= commercial_long and row['RSI'] < rsi_oversold:
                signal_type = 'LONG'
            # Short: Commercial <= 20 AND RSI > 70
            elif row['Commercial_Index'] <= commercial_short and row['RSI'] > rsi_overbought:
                signal_type = 'SHORT'
            
            if signal_type:
                recent_signals.append({
                    'Market': market,
                    'Date': row['Date'],
                    'Signal': signal_type,
                    'COT': round(row['Commercial_Index'], 1),
                    'RSI': round(row['RSI'], 1),
                    'Close': round(row['Close'], 2)
                })
    
    return sorted(recent_signals, key=lambda x: x['Date'], reverse=True)

# Check for signals
signals = check_recent_signals(df6_combined, days=7)

print("=" * 70)
if signals:
    print(f"🔔 ALERT: {len(signals)} TRADING SIGNAL(S) IN THE LAST 7 DAYS!")
    print("=" * 70)
    for sig in signals:
        emoji = "🟢" if sig['Signal'] == 'LONG' else "🔴"
        print(f"{emoji} {sig['Signal']:5} | {sig['Date'].strftime('%Y-%m-%d')} | {sig['Market'][:40]}")
        print(f"         COT: {sig['COT']}, RSI: {sig['RSI']}, Price: ${sig['Close']:,.2f}")
        print("-" * 70)
else:
    print("✓ No new trading signals in the last 7 days")
    print("=" * 70)


🔔 ALERT: 18 TRADING SIGNAL(S) IN THE LAST 7 DAYS!
🟢 LONG  | 2026-03-19 | MICRO GOLD - COMMODITY EXCHANGE INC.
         COT: 100.0, RSI: 19.9, Price: $4,605.70
----------------------------------------------------------------------
🔴 SHORT | 2026-03-19 | SOYBEAN OIL - CHICAGO BOARD OF TRADE
         COT: 0.0, RSI: 80.5, Price: $65.41
----------------------------------------------------------------------
🔴 SHORT | 2026-03-19 | SOYBEANS - CHICAGO BOARD OF TRADE
         COT: 7.8, RSI: 95.8, Price: $1,168.50
----------------------------------------------------------------------
🟢 LONG  | 2026-03-19 | UST 2Y NOTE - CHICAGO BOARD OF TRADE
         COT: 88.4, RSI: 15.2, Price: $103.62
----------------------------------------------------------------------
🔴 SHORT | 2026-03-19 | WHEAT-SRW - CHICAGO BOARD OF TRADE
         COT: 17.4, RSI: 71.5, Price: $608.00
----------------------------------------------------------------------
🟢 LONG  | 2026-03-18 | BRITISH POUND - CHICAGO MERCANTILE EXCHA
    

In [52]:
df6_combined.iloc[-1]

Market                                 WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE
Date                                                           2026-03-19 00:00:00
Close                                                                    96.139999
Open                                                                     99.129997
High                                                                    101.480003
Low                                                                      92.800003
Volume                                                                    157222.0
RSI                                                                      61.122382
YF_Symbol                                                                     CL=F
data_type                                                              daily_price
group                                                                     Energies
OI                                                                            None
OI_I

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go

with open("cot_data.json") as f:
    raw = json.load(f)

oil_df = pd.DataFrame([r for r in raw if r["Market"] == "WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE"
                        and r["data_type"] == "daily_price"])
oil_df["Date"] = pd.to_datetime(oil_df["Date"], unit="ms")
oil_df = oil_df.sort_values("Date").dropna(subset=["Close"])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=oil_df["Date"],
    y=oil_df["Close"],
    mode="lines",
    line=dict(color="#B8860B", width=1.5),
    name="WTI Crude Oil",
))
fig.update_layout(
    title="WTI Crude Oil Price",
    xaxis_title="Year",
    yaxis_title="Price (USD)",
    xaxis=dict(dtick="M12", tickformat="%Y"),
    template="plotly_dark",
    height=500,
    margin=dict(l=60, r=30, t=50, b=50),
)
fig.show()